In [1]:
# Load model directly

import os, sys
# sys.path += ["/teamspace/studios/this_studio"]

import torch
import random
from torch import nn

from typing import Iterator, Tuple, Optional
import json
import numpy as np
import pandas as pd
from gpt2tiny.tokenizer import Tokenizer
import glob
from dataclasses import dataclass
import math
from pytorch_lightning.loggers import MLFlowLogger

from gpt2tiny.reward_v2 import StoryReward
from gpt2tiny.model import GPT2, GPTConfig
# from dataset import PreTokDataset
from gpt2tiny.dataset import (
    RLHFDataset,
    PromptDataset,
    prompt_collator,
)
from gpt2tiny.trainer import TrainingConfig, GRPOGPT2Module
from gpt2tiny.callbacks import LogBestCkptAndPyfuncToMLflow

import torch.distributed as dist
from typing import Iterator, Tuple
from pathlib import Path
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

BASE_DIR = "/teamspace/studios/this_studio/gpt2tiny"
DATA_CACHE_DIR = Path(BASE_DIR) / "data"

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [2]:
trainer_config = TrainingConfig(
    batch_size=16,
    num_workers=4,
    max_iters=200,
    gradient_accumulation_steps=4,
    eval_interval=1,
    log_interval=1,
)

trainer_config

TrainingConfig(learning_rate=0.0006, max_iters=200, weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0, decay_lr=True, warmup_iters=1000, lr_decay_iters=30000, min_lr=6e-05, eval_interval=1, log_interval=1, eval_iters=200, gradient_accumulation_steps=4, batch_size=16, num_workers=4, device='cuda', dtype='bfloat16', compile=True)

In [3]:
tokenizer = Tokenizer(f"{BASE_DIR}/data/tok4096_tinystories.model")

In [4]:
experiment_id = "497642339268382611"
run_id = "f1f7c92703db45ce9b828f08bd386a84"

experiment_id = "497642339268382611"
run_id = "700975a59b844df3be9c7c4837f81dcb"

# experiment_id = "344052319576175981"
# run_id = "16d712577e34431b8a3a93595dcd18bd"

module_path =  glob.glob(str(Path(BASE_DIR) / f"mlruns/{experiment_id}/{run_id}/artifacts/best-step=*.ckpt"))[0]

# reward_weights = {
#     "prompt_adherence": 0.3,
#     "coherence": 0.25,
#     "style": 0.15,
#     "length": 0.15,
#     "safety": 0.0,
#     "fluency": 0.15,
#     "degeneracy_penalty": 0.25,
# }
reward_weights = None

example_prompts = [   
    {   
        'prompt': "Write a short story. The narrative must use 'handle' in its "
                  "role as a noun, include 'map' as a noun, and use the "
                  "adjective 'loud'.",
        'words': [   
            {'word': 'handle', 'pos': 'noun'},
            {'word': 'map', 'pos': 'noun'},
            {'word': 'loud', 'pos': 'adjective'}],
        'features': [],
        'subject': {},
        'feature_phrases': [],
        'word_clause': "use 'handle' in its role as a noun, include 'map' as a "
                       "noun, and use the adjective 'loud'",
        'feature_clause': None,
        'subject_clause': None
    },
    {   
        'prompt': 'Write a short story. Write about a smuggler who delivers '
                  'food during a blackout in a military outpost. The story '
                  "should use 'prince' in its role as a noun. Also ensure that "
                  'there should be a surprising twist in the plot.',
        'words': [
            {'word': 'prince', 'pos': 'noun'}
        ],
        'features': ['Twist'],
        'subject': {   
            'character': 'smuggler',
            'action': 'delivers food during a blackout',
            'place': 'a military outpost',
            'adjective': None,
            'goal': None
        },
        'feature_phrases': ['there should be a surprising twist in the plot'],
        'word_clause': "use 'prince' in its role as a noun",
        'feature_clause': 'there should be a surprising twist in the plot',
        'subject_clause': 'a smuggler who delivers food during a blackout in a '
                          'military outpost.'
    },
    {   
        'prompt': "Write a short story. The narrative must ensure that 'doll' "
                  'is used as a noun. The story should also satisfy the '
                  'following: the plot should contain a meaningful conflict, '
                  'the narrative should contain a clear moral or takeaway, and '
                  'include setup and payoff somewhere in the story.',
        'words': [{'word': 'doll', 'pos': 'noun'}],
        'features': ['Conflict', 'MoralValue', 'Foreshadowing'],
        'subject': {},
        'feature_phrases': [   
            'the plot should contain a meaningful conflict',
            'the narrative should contain a clear moral or '
            'takeaway',
            'include setup and payoff somewhere in the '
            'story'
        ],
        'word_clause': "ensure that 'doll' is used as a noun",
        'feature_clause': 'the plot should contain a meaningful conflict, the '
                          'narrative should contain a clear moral or takeaway, '
                          'and include setup and payoff somewhere in the story',
        'subject_clause': None
    }
]

module = GRPOGPT2Module.load_from_checkpoint(
    module_path,
    weights_only=False,
    prompts=example_prompts,
    # prompts=[
    #     "Write a story about a girl who goes to the mall.",
    #     "Write a story about a dragon and a bear.",
    #     "Write a story about a girl and her cat.",
    # ],
    max_seq_len = 248,
    num_gen=10,
    reward_weights=reward_weights,
    lr=3e-5,
    temperature = 1.5,

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# module = torch.compile(module) 

In [6]:
device = module.device

In [7]:
device

device(type='cuda', index=0)

In [8]:
train_dataloader= DataLoader(
    PromptDataset(
        data_dir=DATA_CACHE_DIR / f"TinyStories_prompts_v2",
        split="train",
        weights="Balanced",
    ),
    collate_fn=lambda batch: prompt_collator(batch),
    batch_size=trainer_config.batch_size,
    num_workers=trainer_config.num_workers,
)

eval_dataloader= DataLoader(
    PromptDataset(
        data_dir=DATA_CACHE_DIR / f"TinyStories_prompts_v2",
        split="validation",
        weights="Balanced",
    ),
    collate_fn=lambda batch: prompt_collator(batch),
    batch_size=trainer_config.batch_size,
    num_workers=trainer_config.num_workers,
)

In [9]:
trainer_config.max_iters

200

In [10]:
experiment_name = "rlhf_tests_v2"

run_name = "tinystories-grpo"
run_name += "-" + pd.Timestamp.now().strftime("%Y-%m-%d-%H%M%S")

model_name = "GPT2TinyStoriesGRPO_v2"


mlf_logger = MLFlowLogger(
    experiment_name=experiment_name,
    # tracking_uri="http://127.0.0.1:5000",
    tracking_uri=f"file:{BASE_DIR}/mlruns",  # Colab-local (ephemeral) filesystem
    run_name=run_name,
)

# crucial change: set dirpath to mlf_logger.log_dir
# this makes ModelCheckpoint save files into the local directory that MLFlowLogger monitors for artifacts
checkpoint_cb = ModelCheckpoint(
    monitor="val_avg_reward",
    mode="max",
    save_top_k=1,
    dirpath=f"{BASE_DIR}/mlruns/{mlf_logger.experiment_id}/{mlf_logger.run_id}/artifacts/",
    filename="best-{step}-{val_avg_reward:.4f}",
)


trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    max_steps=trainer_config.max_iters,
    val_check_interval=trainer_config.gradient_accumulation_steps * trainer_config.eval_interval,
    limit_val_batches=1,
    logger=mlf_logger,
    callbacks=[
        checkpoint_cb,
        LogBestCkptAndPyfuncToMLflow(module_cls=GRPOGPT2Module, register_name=model_name),
    ],
    log_every_n_steps=trainer_config.log_interval,
    accumulate_grad_batches=trainer_config.gradient_accumulation_steps,
    gradient_clip_val=trainer_config.grad_clip,
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


In [11]:
trainer.fit(module, train_dataloader, eval_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


lr = 3e-05


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model      │ GPT2           │ 27.5 M │ train │     0 │
│ 1 │ ref_policy │ GRPOGPT2Module │ 27.5 M │ eval  │     0 │
└───┴────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 27.5 M                                                                                           
Non-trainable params: 27.5 M                                                                                       
Total params: 55.1 M                                                                                               
Total estimated model params size (MB): 220                                                                        
Modules in train mode: 112                                                                                         
Modules in eval mode: 113                                                                                          
Total FLOPs: 0

...

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:79: Trying 
to infer the `batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any 
miscalculations, use `self.log(..., batch_size=batch_size)`.

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py:534: Found 
113 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this 
is intentional, you can ignore this warning.


Detected KeyboardInterrupt, attempting graceful shutdown ...


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/home/zeus/min